# Bài toán 4: Dự đoán Thời gian Chuyến xe (Trip Duration)

## Mục đích
- Dự đoán thời gian chuyến xe (trip_duration) dựa trên các đặc trưng
- Sử dụng output từ các bài toán trước: `location_cluster`, `behavior_cluster`
- So sánh nhiều mô hình ML Regressor và chọn mô hình tốt nhất

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import gc, time
try:
    import psutil
    PSUTIL_AVAILABLE = True
except Exception:
    PSUTIL_AVAILABLE = False

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, RandomizedSearchCV, HalvingRandomSearchCV
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error,
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

## 2. Load dữ liệu

In [2]:
df = pd.read_csv('../data/processed/yellow_tripdata_2016-03_processed.csv')

print(f"Dữ liệu loaded: {df.shape}")
print(f"Các cột trong dataset: {df.columns.tolist()}")
print(f"\nTrip duration stats:")
print(df['trip_duration'].describe())

Dữ liệu loaded: (11411918, 25)
Các cột trong dataset: ['Unnamed: 0', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'fare_amount', 'trip_duration', 'speed', 'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'time_of_day', 'is_peak_hour', 'is_credit_card', 'total_income', 'location_cluster', 'fare_per_miles', 'fare_per_min', 'traffic_index', 'traffic_level', 'behavior_cluster']

Trip duration stats:
count    1.141192e+07
mean     7.471010e+02
std      4.733584e+02
min      6.100000e+01
25%      3.900000e+02
50%      6.350000e+02
75%      9.950000e+02
max      2.655000e+03
Name: trip_duration, dtype: float64


## 3. Chuẩn bị Features cho Regressor

Sau khi thực hiện các bài toán trước, nhóm đã có đầy đủ được các feature để dùng cho bài toán dự đoán thời gian `trip_duration`

In [7]:
feature_cols = [
    'trip_distance',
    'pickup_longitude', 
    'pickup_latitude',
    'dropoff_longitude', 
    'dropoff_latitude',
    'pickup_hour',
    # 'fare_per_min',
    'is_peak_hour',
    'location_cluster',
    'behavior_cluster',
    'traffic_level'
]

feature_scales = [
    'trip_distance',
    'pickup_longitude', 
    'pickup_latitude',
    'dropoff_longitude', 
    'dropoff_latitude',
    # 'fare_per_min'
]

# Kiểm tra features có sẵn
available_features = [f for f in feature_cols if f in df.columns]
print(f"Features sử dụng ({len(available_features)}): {available_features}")

X = df[available_features].copy()
y = df['trip_duration'].copy()

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeature stats:")
print(X.describe())

Features sử dụng (10): ['trip_distance', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'is_peak_hour', 'location_cluster', 'behavior_cluster', 'traffic_level']

X shape: (11411918, 10)
y shape: (11411918,)

Feature stats:
       trip_distance  pickup_longitude  pickup_latitude  dropoff_longitude  \
count   1.141192e+07      1.141192e+07     1.141192e+07       1.141192e+07   
mean    2.433433e+00     -7.397761e+01     4.075276e+01      -7.397535e+01   
std     2.267391e+00      2.712710e-02     2.407487e-02       2.747731e-02   
min     1.100000e-01     -7.414575e+01     4.057748e+01      -7.414951e+01   
25%     1.000000e+00     -7.399213e+01     4.073857e+01      -7.399142e+01   
50%     1.650000e+00     -7.398216e+01     4.075448e+01      -7.398017e+01   
75%     2.900000e+00     -7.396909e+01     4.076815e+01      -7.396438e+01   
max     1.326000e+01     -7.370239e+01     4.091751e+01      -7.370040e+01   

       dropoff_latitude   

## 4. Train/Test Split

Nhóm phân dữ liệu ra làm 3 phần: train, test, valid

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Split temp (30%) into validation (50% of temp = 15%) and test (50% of temp = 15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Training set: {X_train.shape} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTarget distribution:")
print(f"Train - Mean: {y_train.mean():.2f}s, Std: {y_train.std():.2f}s")
print(f"Val   - Mean: {y_val.mean():.2f}s, Std: {y_val.std():.2f}s")
print(f"Test  - Mean: {y_test.mean():.2f}s, Std: {y_test.std():.2f}s")

Training set: (7988342, 10) (70.0%)
Validation set: (1711788, 10) (15.0%)
Test set: (1711788, 10) (15.0%)

Target distribution:
Train - Mean: 747.26s, Std: 473.44s
Val   - Mean: 746.85s, Std: 473.05s
Test  - Mean: 746.59s, Std: 473.29s


## 5. Định nghĩa các mô hình Regressor


Tiêu chí lựa chọn mô hình là phải hoạt động tốt trên dữ liệu lớn của taxi dataset với hiệu suất cao và thời gian train nhanh chóng.

1. **LightGBM**: Mô hình boosting tối ưu cho dữ liệu lớn, có tốc độ train nhanh và xử lý tốt các quan hệ phi tuyến trong dữ liệu taxi.
2. **CatBoost**: Mô hình boosting mạnh trong xử lý các feature dạng categorical và có khả năng giảm overfitting tốt.
3. **XGBoost**: Mô hình boosting phổ biến với độ chính xác cao và khả năng học tốt các mối quan hệ phức tạp trong dữ liệu.

In [9]:
# Định nghĩa các mô hình với thông tin về scaling requirement
# Tree-based models (LightGBM, CatBoost, XGBoost) do NOT require feature scaling
models = {
    'LightGBM': (LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, 
                                num_leaves=31, random_state=42, n_jobs=-1, verbose=-1), False),
    'CatBoost': (CatBoostRegressor(iterations=100, max_depth=6, learning_rate=0.1,
                                   random_state=42, thread_count=-1, verbose=False), False),
    'XGBoost': (XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                             random_state=42, n_jobs=-1, tree_method='hist', verbosity=0), False)
}

print(f"{len(models)} mô hình đã được định nghĩa:")
for name in models.keys():
    print(f"  - {name}")

3 mô hình đã được định nghĩa:
  - LightGBM
  - CatBoost
  - XGBoost


## 6. Chuẩn bị Scaler

In [10]:
scaler = StandardScaler()
X_train[feature_scales] = scaler.fit_transform(X_train[feature_scales])
X_val[feature_scales] = scaler.transform(X_val[feature_scales])
X_test[feature_scales] = scaler.transform(X_test[feature_scales])

print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape: {X_val.shape}")
print(f"   X_test shape: {X_test.shape}")

   X_train shape: (7988342, 10)
   X_val shape: (1711788, 10)
   X_test shape: (1711788, 10)


## 7. Training và Evaluation Models

In [11]:
results = {}

print("="*80)
print("TRAINING VÀ ĐÁNH GIÁ CÁC MÔ HÌNH")
print("="*80)

for name, (model, needs_scaling) in models.items():
    print(f"\n Training {name}...", end=' ', flush=True)
    
    model.fit(X_train, y_train)
    
    # Dự đoán
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    val_mse = mean_squared_error(y_val, y_pred_val)
    val_rmse = np.sqrt(val_mse)
    val_mae = mean_absolute_error(y_val, y_pred_val)
    val_r2 = r2_score(y_val, y_pred_val)
    val_mape = mean_absolute_percentage_error(y_val, y_pred_val)
    
    results[name] = {
        'model': model,
        'scaled': needs_scaling,
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'val_rmse': val_rmse,
        'val_mae': val_mae,
        'val_r2': val_r2,
        'val_mape': val_mape,
        'y_pred_val': y_pred_val,
        'y_pred_test': y_pred_test
    }
    
    print(f"")
    print(f"  Val  - RMSE: {val_rmse:.2f}s | MAE: {val_mae:.2f}s | R²: {val_r2:.4f} | MAPE: {val_mape:.2%}")

print("\n" + "="*80)
print("Tất cả mô hình đã được huấn luyện!")
print("="*80)

TRAINING VÀ ĐÁNH GIÁ CÁC MÔ HÌNH

 Training LightGBM... 
  Val  - RMSE: 135.06s | MAE: 87.63s | R²: 0.9185 | MAPE: 12.63%

 Training CatBoost... 
  Val  - RMSE: 139.35s | MAE: 90.37s | R²: 0.9132 | MAPE: 13.16%

 Training XGBoost... 
  Val  - RMSE: 133.71s | MAE: 86.33s | R²: 0.9201 | MAPE: 12.35%

Tất cả mô hình đã được huấn luyện!


## 8. So sánh kết quả và chọn mô hình tốt nhất

In [12]:
# Tạo dataframe để so sánh các mô hình
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    # 'Train RMSE': [results[m]['train_rmse'] for m in results.keys()],
    'Val RMSE': [results[m]['val_rmse'] for m in results.keys()],
    'Val MAE': [results[m]['val_mae'] for m in results.keys()],
    'Val R²': [results[m]['val_r2'] for m in results.keys()],
    'Val MAPE': [results[m]['val_mape'] for m in results.keys()],
    # 'Test RMSE': [results[m]['test_rmse'] for m in results.keys()],
    # 'Test MAE': [results[m]['test_mae'] for m in results.keys()],
    # 'Test R²': [results[m]['test_r2'] for m in results.keys()],
    # 'Test MAPE': [results[m]['test_mape'] for m in results.keys()],
})

# Sắp xếp theo Val RMSE
comparison_df = comparison_df.sort_values('Val RMSE').reset_index(drop=True)

print("\n BẢNG SO SÁNH KẾT QUẢ CÁC MÔ HÌNH (VALIDATION SET)")
print("="*140)
print(comparison_df.to_string(index=False))
print("="*140)

# Chọn mô hình tốt nhất
best_model_name = comparison_df.iloc[0]['Model']
best_val_rmse = comparison_df.iloc[0]['Val RMSE']
best_val_r2 = comparison_df.iloc[0]['Val R²']
# best_test_rmse = comparison_df.iloc[0]['Test RMSE']
# best_test_r2 = comparison_df.iloc[0]['Test R²']

print(f"\n MÔ HÌNH TỐT NHẤT (Validation set): {best_model_name}")
print(f"  Val RMSE: {best_val_rmse:.2f}s | Val R²: {best_val_r2:.4f}")
# print(f"  Test RMSE: {best_test_rmse:.2f}s | Test R²: {best_test_r2:.4f}")


 BẢNG SO SÁNH KẾT QUẢ CÁC MÔ HÌNH (VALIDATION SET)
   Model   Val RMSE   Val MAE   Val R²  Val MAPE
 XGBoost 133.713687 86.330908 0.920103  0.123484
LightGBM 135.062858 87.630595 0.918483  0.126338
CatBoost 139.345798 90.373108 0.913231  0.131633

 MÔ HÌNH TỐT NHẤT (Validation set): XGBoost
  Val RMSE: 133.71s | Val R²: 0.9201


## 9. Hyperparameter Tuning cho các Model

In [13]:
import optuna

sample_size = 500000

sample_indices = np.random.choice(
    len(X_train),
    sample_size,
    replace=False
)

X_train_tune = X_train.iloc[sample_indices]
y_train_tune = y_train.iloc[sample_indices]

def objective_lgbm(trial):

    params = {
    'n_estimators': trial.suggest_int('n_estimators', 100, 300),
    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15),
    'max_depth': trial.suggest_int('max_depth', 4, 10),
    'num_leaves': trial.suggest_int('num_leaves', 20, 64),
    'subsample': trial.suggest_float('subsample', 0.7, 1.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),

    'min_child_samples': 20,
    'min_data_in_leaf': 20,

    'random_state': 42,
    'n_jobs': -1,

    'device': 'cpu'
}

    model = LGBMRegressor(**params)

    model.fit(X_train_tune, y_train_tune)

    y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    trial.set_user_attr("rmse", rmse)
    trial.set_user_attr("mae", mae)
    trial.set_user_attr("r2", r2)

    return rmse

# =====================================================
# XGBOOST
# =====================================================

def objective_xgb(trial):

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),

        'random_state': 42,
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'device': 'cuda'
    }

    model = XGBRegressor(**params)  # Lấy mô hình XGBoost đã định nghĩa trước đó

    model.fit(X_train_tune, y_train_tune)

    y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    trial.set_user_attr("rmse", rmse)
    trial.set_user_attr("mae", mae)
    trial.set_user_attr("r2", r2)

    return rmse

    

# =====================================================
# CATBOOST
# =====================================================

def objective_cat(trial):

    params = {
        'iterations': trial.suggest_int('iterations', 100, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),

        'random_state': 42,
        'verbose': False,
        'task_type': 'GPU',
        'devices': '0'
    }

    model = CatBoostRegressor(**params)  # Lấy mô hình CatBoost đã định nghĩa trước đó

    model.fit(X_train_tune, y_train_tune)

    y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    trial.set_user_attr("rmse", rmse)
    trial.set_user_attr("mae", mae)
    trial.set_user_attr("r2", r2)

    return rmse

# =====================================================
# RUN OPTUNA
# =====================================================

print("=" * 50)
print("TUNING LIGHTGBM")
print("=" * 50)

study_lgbm = optuna.create_study(direction='minimize')

study_lgbm.optimize(objective_lgbm, n_trials=20)

print("Best LightGBM Score:", study_lgbm.best_value)
print("Best LightGBM Params:")
print(study_lgbm.best_params)
print("\nBest LightGBM Metrics:")
for key, value in study_lgbm.best_trial.user_attrs.items():
    print(f"  {key}: {value}")

# =====================================================

print("=" * 50)
print("TUNING XGBOOST")
print("=" * 50)

study_xgb = optuna.create_study(direction='minimize')

study_xgb.optimize(objective_xgb, n_trials=20)

print("Best XGBoost Score:", study_xgb.best_value)
print("Best XGBoost Params:")
print(study_xgb.best_params)
print("\nBest XGBoost Metrics:")
for key, value in study_xgb.best_trial.user_attrs.items():
    print(f"  {key}: {value}")

# =====================================================

print("=" * 50)
print("TUNING CATBOOST")
print("=" * 50)

study_cat = optuna.create_study(direction='minimize')

study_cat.optimize(objective_cat, n_trials=20)

print("Best CatBoost Score:", study_cat.best_value)
print("Best CatBoost Params:")
print(study_cat.best_params)
print("\nBest CatBoost Metrics:")

for key, value in study_cat.best_trial.user_attrs.items():
    print(f"  {key}: {value}")

[I 2026-05-13 20:38:40,349] A new study created in memory with name: no-name-9d9e0ddb-cb4e-49e2-b931-34fe6847aed8


TUNING LIGHTGBM


[I 2026-05-13 20:38:45,261] Trial 0 finished with value: 132.75362262151648 and parameters: {'n_estimators': 281, 'learning_rate': 0.04574097291535694, 'max_depth': 7, 'num_leaves': 49, 'subsample': 0.8115125155819345, 'colsample_bytree': 0.934370630330438}. Best is trial 0 with value: 132.75362262151648.
[I 2026-05-13 20:38:49,807] Trial 1 finished with value: 132.47435566858036 and parameters: {'n_estimators': 254, 'learning_rate': 0.058281179823180644, 'max_depth': 8, 'num_leaves': 57, 'subsample': 0.8516827971241339, 'colsample_bytree': 0.7106819014749697}. Best is trial 1 with value: 132.47435566858036.
[I 2026-05-13 20:38:52,134] Trial 2 finished with value: 132.81496484262547 and parameters: {'n_estimators': 162, 'learning_rate': 0.11315853125104787, 'max_depth': 8, 'num_leaves': 28, 'subsample': 0.842126786213196, 'colsample_bytree': 0.8631619296935888}. Best is trial 1 with value: 132.47435566858036.
[I 2026-05-13 20:38:54,861] Trial 3 finished with value: 130.5398338843597 an

Best LightGBM Score: 130.29087907353278
Best LightGBM Params:
{'n_estimators': 193, 'learning_rate': 0.14720735190175888, 'max_depth': 10, 'num_leaves': 64, 'subsample': 0.7055491415067423, 'colsample_bytree': 0.9749214167468082}

Best LightGBM Metrics:
  rmse: 130.29087907353278
  mae: 83.63624257548958
  r2: 0.9241411869872984
TUNING XGBOOST


[I 2026-05-13 20:39:49,256] Trial 0 finished with value: 131.18152002356243 and parameters: {'n_estimators': 231, 'learning_rate': 0.13308878390245293, 'max_depth': 6, 'subsample': 0.8199707812143231, 'colsample_bytree': 0.9254761235162712}. Best is trial 0 with value: 131.18152002356243.
[I 2026-05-13 20:39:53,608] Trial 1 finished with value: 130.9665679815617 and parameters: {'n_estimators': 189, 'learning_rate': 0.1840422574758724, 'max_depth': 8, 'subsample': 0.8453408958353931, 'colsample_bytree': 0.8998191553367147}. Best is trial 1 with value: 130.9665679815617.
[I 2026-05-13 20:40:00,910] Trial 2 finished with value: 129.37448767143417 and parameters: {'n_estimators': 226, 'learning_rate': 0.07728641867874919, 'max_depth': 10, 'subsample': 0.9706078567871194, 'colsample_bytree': 0.9113598815205118}. Best is trial 2 with value: 129.37448767143417.
[I 2026-05-13 20:40:10,195] Trial 3 finished with value: 132.53420507884056 and parameters: {'n_estimators': 381, 'learning_rate': 0

Best XGBoost Score: 129.37448767143417
Best XGBoost Params:
{'n_estimators': 226, 'learning_rate': 0.07728641867874919, 'max_depth': 10, 'subsample': 0.9706078567871194, 'colsample_bytree': 0.9113598815205118}

Best XGBoost Metrics:
  rmse: 129.37448767143417
  mae: 82.34472312210164
  r2: 0.925204529185477
TUNING CATBOOST


[I 2026-05-13 20:42:21,162] Trial 0 finished with value: 141.14955692067966 and parameters: {'iterations': 166, 'learning_rate': 0.10271374606883027, 'depth': 4, 'l2_leaf_reg': 9.445773366407046}. Best is trial 0 with value: 141.14955692067966.
[I 2026-05-13 20:42:24,253] Trial 1 finished with value: 135.0956244256923 and parameters: {'iterations': 388, 'learning_rate': 0.17807173503219137, 'depth': 4, 'l2_leaf_reg': 5.53641731166865}. Best is trial 1 with value: 135.0956244256923.
[I 2026-05-13 20:42:26,854] Trial 2 finished with value: 142.03589483962318 and parameters: {'iterations': 114, 'learning_rate': 0.03727656610784322, 'depth': 10, 'l2_leaf_reg': 7.7268125950589726}. Best is trial 1 with value: 135.0956244256923.
[I 2026-05-13 20:42:34,702] Trial 3 finished with value: 130.83654074841274 and parameters: {'iterations': 374, 'learning_rate': 0.07197867137494236, 'depth': 10, 'l2_leaf_reg': 4.8270833383919545}. Best is trial 3 with value: 130.83654074841274.
[I 2026-05-13 20:42:

Best CatBoost Score: 130.08667209718894
Best CatBoost Params:
{'iterations': 341, 'learning_rate': 0.16301859559981902, 'depth': 10, 'l2_leaf_reg': 8.644177296751968}

Best CatBoost Metrics:
  rmse: 130.08667209718894
  mae: 83.2383325095681
  r2: 0.9243787901008519


## 10. So Sánh Kết Quả Trước và Sau Tuning

In [ ]:
# =====================================================
# TẠO DATAFRAME SO SÁNH TRƯỚC & SAU TUNING
# =====================================================

comparison_tuning = []

after_results = [
    {
        'Model': 'CatBoost',
        **study_cat.best_trial.user_attrs
    },

    {
        'Model': 'XGBoost',
        **study_xgb.best_trial.user_attrs
    },

    {
        'Model': 'LightGBM',
        **study_lgbm.best_trial.user_attrs
    }
]

for model_name in models.keys():

    # =================================================
    # BEFORE TUNING
    # =================================================

    before = comparison_df[
        comparison_df['Model'] == model_name
    ].iloc[0]

    # =================================================
    # AFTER TUNING
    # =================================================

    after = next(
        (
            item for item in after_results
            if item['Model'] == model_name
        ),
        None
    )

    # =================================================
    # APPEND RESULT
    # =================================================

    comparison_tuning.append({

        'Model': model_name,

        'Before Val RMSE': before['Val RMSE'],
        'After Val RMSE': after['rmse'],

        'Val RMSE Improvement (%)':
        (
            (
                before['Val RMSE'] - after['rmse']
            )
            / before['Val RMSE']
        ) * 100,

        'Before Val R²': before['Val R²'],
        'After Val R²': after['r2'],

        'Val R² Improvement':
        (
            after['r2'] - before['Val R²']
        ),
    })

# =====================================================
# DATAFRAME
# =====================================================

comparison_tuning_df = pd.DataFrame(comparison_tuning)

print("\nBẢNG SO SÁNH KẾT QUẢ TRƯỚC VÀ SAU TUNING")
print("=" * 160)

print(comparison_tuning_df.to_string(index=False))

print("=" * 160)

# =====================================================
# BEST MODEL
# =====================================================

best_row = comparison_tuning_df.loc[
    comparison_tuning_df['After Val RMSE'].idxmin()
]

print(f"\n🏆 MÔ HÌNH TỐT NHẤT SAU TUNING: {best_row['Model']}")

print(f"\nValidation Set:")
print(f"  - Val RMSE: {best_row['After Val RMSE']:.2f}s")
print(f"  - Val R²: {best_row['After Val R²']:.4f}")
print(f"  - RMSE Improvement: {best_row['Val RMSE Improvement (%)']:.2f}%")


BẢNG SO SÁNH KẾT QUẢ TRƯỚC VÀ SAU TUNING
   Model  Before Val RMSE  After Val RMSE  Val RMSE Improvement (%)  Before Val R²  After Val R²  Val R² Improvement
LightGBM       135.062858      130.290879                  3.533154       0.918483      0.924141            0.005659
CatBoost       139.345798      130.086672                  6.644711       0.913231      0.924379            0.011148
 XGBoost       133.713687      129.374488                  3.245142       0.920103      0.925205            0.005101

🏆 MÔ HÌNH TỐT NHẤT SAU TUNING: XGBoost

Validation Set:
  - Val RMSE: 129.37s
  - Val R²: 0.9252
  - RMSE Improvement: 3.25%


## 11. Lấy best_models dự đoán trên tập test

In [15]:
best_lightgbm = LGBMRegressor(**study_lgbm.best_params, random_state=42, n_jobs=-1, device="cpu")

best_xgb = XGBRegressor(**study_xgb.best_params, random_state=42, use_label_encoder=False, eval_metric='mlogloss', tree_method='hist', device='gpu')

best_cat = CatBoostRegressor(**study_cat.best_params, random_state=42, verbose=False, task_type='GPU', devices='0')

best_lightgbm.fit(X_train, y_train)
best_xgb.fit(X_train, y_train)
best_cat.fit(X_train, y_train)

y_pred_lgbm = best_lightgbm.predict(X_test)
y_pred_xgb = best_xgb.predict(X_test)
y_pred_cat = best_cat.predict(X_test)

rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
rmse_cat = np.sqrt(mean_squared_error(y_test, y_pred_cat))

mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mae_cat = mean_absolute_error(y_test, y_pred_cat)

r2_core_macro_lgbm = r2_score(y_test, y_pred_lgbm)
r2_core_macro_xgb = r2_score(y_test, y_pred_xgb)
r2_core_macro_cat = r2_score(y_test, y_pred_cat)



In [16]:
comparison_df = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'RMSE': [rmse_lgbm, rmse_xgb, rmse_cat],
    'MAE': [mae_lgbm, mae_xgb, mae_cat],
    'R2': [r2_core_macro_lgbm, r2_core_macro_xgb, r2_core_macro_cat]
}).sort_values('RMSE', ascending=True).reset_index(drop=True)

best_model_name = comparison_df.iloc[0]['Model']
best_model = {
    'LightGBM': best_lightgbm,
    'XGBoost': best_xgb,
    'CatBoost': best_cat
}[best_model_name]


print("Kết quả dự đoán trên tập test:")

print(comparison_df.to_string(index=False))
print(f"\n🏆Mô hình tốt nhất trên test set: {best_model_name}")


Kết quả dự đoán trên tập test:
   Model       RMSE       MAE       R2
 XGBoost 126.123710 80.517221 0.928987
CatBoost 128.096166 82.060671 0.926748
LightGBM 128.504323 82.597883 0.926281

🏆Mô hình tốt nhất trên test set: XGBoost


## 12. Dự đoán Trip Duration cho toàn bộ Dataset

In [18]:
# COPY DATA
X_full = df[feature_cols].copy()

# SCALE
X_full[feature_scales] = scaler.transform(
    X_full[feature_scales]
)

# PREDICT
y_pred_full = best_xgb.predict(X_full)

# SAVE PREDICTION
df['predicted_trip_duration'] = y_pred_full

# =====================================================
# DISPLAY
# =====================================================

print(f"✅ Dự đoán hoàn tất cho {len(df):,} records!")

print("\nPredicted trip_duration statistics:")
print(df['predicted_trip_duration'].describe())

print(f"\nMean comparison:")
print(f"  Thực tế  : {df['trip_duration'].mean():.2f}s")
print(f"  Dự đoán  : {df['predicted_trip_duration'].mean():.2f}s")

print("\nSample predictions:")

display(
    df[
        feature_cols +
        ['trip_duration', 'predicted_trip_duration']
    ].head(10)
)

✅ Dự đoán hoàn tất cho 11,411,918 records!

Predicted trip_duration statistics:
count    1.141192e+07
mean     7.471000e+02
std      4.671212e+02
min      6.266455e+01
25%      3.906214e+02
50%      6.358725e+02
75%      9.952730e+02
max      2.673340e+03
Name: predicted_trip_duration, dtype: float64

Mean comparison:
  Thực tế  : 747.10s
  Dự đoán  : 747.10s

Sample predictions:


,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,pickup_hour,fare_per_min,is_peak_hour,location_cluster,behavior_cluster,traffic_level,trip_duration,predicted_trip_duration
0,2.50,-73.976746,40.765152,-74.004265,40.746128,0,1.136842,0,0,2,0,475.0,507.359802
1,2.90,-73.983482,40.767925,-74.005943,40.733166,0,0.990991,0,0,2,0,666.0,673.084412
2,6.20,-73.788773,40.647758,-73.829208,40.712345,0,1.277259,0,-1,1,0,963.0,923.292053
3,0.70,-73.958221,40.764641,-73.967896,40.762901,0,1.103679,0,0,0,2,299.0,298.075226
4,7.18,-73.985779,40.741192,-73.946350,40.797878,0,0.975779,0,0,1,0,1445.0,1500.122925
5,0.54,-73.988426,40.764160,-73.992393,40.758224,0,1.967213,0,0,3,0,122.0,112.934715
6,1.70,-73.969818,40.797428,-73.943771,40.796200,0,1.027837,0,0,3,0,467.0,475.198334
7,1.10,-73.953804,40.788128,-73.971550,40.795238,0,1.803279,0,0,2,0,183.0,170.793808
8,2.10,-73.976089,40.752171,-73.987450,40.770782,0,0.947368,0,0,2,0,570.0,582.670837
9,8.54,-74.002068,40.719120,-73.952118,40.811241,0,1.118012,0,0,1,0,1449.0,1450.006714


## 13. Lưu các best model

In [17]:
import joblib

joblib.dump(best_xgb, './model/trip_duration/best_xgb_model.pkl')

joblib.dump(best_cat, './model/trip_duration/best_cat_model.pkl')

joblib.dump(best_lightgbm, './model/trip_duration/best_lightgbm_model.pkl')

['./model/trip_duration/best_lightgbm_model.pkl']

## Load model

In [ ]:
loaded_xgb = joblib.load('./model/trip_duration/best_xgb_model.pkl')
loaded_cat = joblib.load('./model/trip_duration/best_cat_model.pkl')
loaded_lightgbm = joblib.load('./model/trip_duration/best_lightgbm_model.pkl')